# 2단계: 학습 전략 Ablation Test

**목적**: B04 설정의 각 요소가 실제로 효과가 있는지 검증

```
B04 = 차등 학습률 + Label Smoothing + Mean Pooling
       ↓ 빼봄          ↓ 빼봄           ↓ 바꿔봄
   성능 떨어지면?     성능 떨어지면?     성능 떨어지면?
   → 필요함          → 필요함          → Mean이 나음
```

**고정**: klue/roberta-base (1단계 최적 모델)

| 그룹 | 실험 | 변경 변수 |
|---|---|---|
| LR | M2-A / M2-B / M2-C | 단일 2e-5 / 차등 4e-6·2e-5 / 초보수적 2e-6·2e-5 |
| LS | M2-D / M2-E | Label Smoothing 0.0 / 0.1 |
| Pooling | M2-F / M2-G | Mean Pooling / [CLS] Pooling |

In [ ]:
# ============================================================
# Cell 1: 환경 설정
# ============================================================
import pandas as pd
import numpy as np
import re
import os
import time
import gc
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Colab 한글 폰트
import subprocess
subprocess.run(['apt-get', '-y', 'install', 'fonts-nanum'], capture_output=True)
import matplotlib.font_manager as fm
fm._rebuild()
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ============================================================
# Cell 2: 데이터 로드
# ============================================================
import os
if not os.path.exists('/content/DLThon'):
    !git clone https://github.com/seongyeon-h/DLThon.git /content/DLThon

DATA_DIR = '/content/DLThon/data'
train_df = pd.read_csv(f'{DATA_DIR}/train_final.csv')
val_df = pd.read_csv(f'{DATA_DIR}/val_final.csv')
test_df_raw = pd.read_csv(f'{DATA_DIR}/test.csv')

label2id = {'갈취 대화':0, '기타 괴롭힘 대화':1, '직장 내 괴롭힘 대화':2, '협박 대화':3, '일반 대화':4}
id2label = {v:k for k,v in label2id.items()}

def preprocess(text):
    if not isinstance(text, str): return ''
    return re.sub(r'\s+', ' ', text.replace('\n', ' ')).strip()

train_df['text'] = train_df['conversation'].apply(preprocess)
train_df['label'] = train_df['class'].map(label2id)
val_df['text'] = val_df['conversation'].apply(preprocess)
val_df['label'] = val_df['class'].map(label2id)
test_df_raw['text'] = test_df_raw['conversation'].apply(lambda x: preprocess(str(x).replace('"','')))

train_texts, train_labels = train_df['text'].tolist(), train_df['label'].tolist()
val_texts, val_labels = val_df['text'].tolist(), val_df['label'].tolist()
test_texts = test_df_raw['text'].tolist()

print(f'Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}')

In [ ]:
# ============================================================
# Cell 3: 고정값 + 실험 설정 정의
# ============================================================
MODEL_NAME = 'klue/roberta-base'  # 1단계 최적 모델 고정
MAX_LEN = 256
BATCH_SIZE = 32
EPOCHS = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
DROPOUT_RATE = 0.3

# 실험 목록 (M2-B는 1단계 M1-A와 동일하므로 스킵, 결과 참조: Val F1=0.9299, Test 일반=74건)
EXPERIMENTS_DEDUP = [
    # (실험ID, 설명, backbone_lr, head_lr, label_smoothing, pooling)
    ('M2-A', '단일 LR 2e-5',      2e-5, 2e-5, 0.0, 'mean'),
    ('M2-C', '초보수적 2e-6/2e-5', 2e-6, 2e-5, 0.0, 'mean'),
    ('M2-E', 'LS=0.1',            4e-6, 2e-5, 0.1, 'mean'),
    ('M2-G', '[CLS] Pooling',     4e-6, 2e-5, 0.0, 'cls'),
]

# M2-B 기준선 (1단계 M1-A 결과로 대체)
M2_B_REFERENCE = {'exp_id': 'M2-B', 'desc': '차등 LR 4e-6/2e-5 (기준)',
                   'val_f1': 0.9299, 'normal_val_f1': 0.9949, 'test_normal': 74}

print(f'모델: {MODEL_NAME} (고정)')
print(f'실험 수: {len(EXPERIMENTS_DEDUP)}종 + M2-B 기준(M1-A 결과 참조)')
print(f'M2-B 기준: Val F1={M2_B_REFERENCE["val_f1"]}, Test 일반={M2_B_REFERENCE["test_normal"]}건')
print()
for eid, desc, blr, hlr, ls, pool in EXPERIMENTS_DEDUP:
    print(f'  {eid}: {desc} | backbone_lr={blr}, head_lr={hlr}, LS={ls}, pool={pool}')

In [ ]:
# ============================================================
# Cell 4: Dataset + 모델 (Mean/CLS 풀링 선택) + 학습/검증 함수
# ============================================================

class ConversationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}


class ConversationClassifier(nn.Module):
    def __init__(self, model_name, num_classes=5, dropout_rate=0.3, pooling='mean'):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.pooling = pooling  # 'mean' or 'cls'
        h = self.backbone.config.hidden_size
        self.layer_norm = nn.LayerNorm(h)
        self.fc1 = nn.Linear(h, 256)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        if self.pooling == 'mean':
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        else:  # cls
            pooled = out.last_hidden_state[:, 0, :]  # [CLS] 토큰
        pooled = self.layer_norm(pooled)
        return self.fc2(self.dropout(self.activation(self.fc1(pooled))))


def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss, preds, trues = 0, [], []
    for batch in loader:
        ids, mask, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds.extend(logits.argmax(-1).cpu().numpy())
        trues.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(trues, preds, average='macro')


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds, trues = 0, [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
            logits = model(ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds.extend(logits.argmax(-1).cpu().numpy())
            trues.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(trues, preds, average='macro'), preds, trues


print('Dataset, Model(Mean/CLS 선택), 학습/검증 함수 정의 완료')

In [ ]:
# ============================================================
# Cell 5: 5종 자동 순회 학습
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = ConversationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = ConversationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = ConversationDataset(test_texts, [0]*len(test_texts), tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_results = []

for exp_id, desc, backbone_lr, head_lr, ls, pooling in EXPERIMENTS_DEDUP:
    print(f'\n{"="*70}')
    print(f'[{exp_id}] {desc}')
    print(f'  backbone_lr={backbone_lr}, head_lr={head_lr}, LS={ls}, pooling={pooling}')
    print(f'{"="*70}')
    
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    
    # 모델 생성 (풀링 방식 선택)
    model = ConversationClassifier(MODEL_NAME, len(label2id), DROPOUT_RATE, pooling=pooling).to(device)
    
    # 차등 학습률
    head_p, backbone_p = [], []
    for name, param in model.named_parameters():
        if any(k in name for k in ['fc1', 'fc2', 'layer_norm']):
            head_p.append(param)
        else:
            backbone_p.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_p, 'lr': backbone_lr, 'weight_decay': WEIGHT_DECAY},
        {'params': head_p, 'lr': head_lr, 'weight_decay': WEIGHT_DECAY},
    ])
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    criterion = nn.CrossEntropyLoss(label_smoothing=ls)
    
    best_f1, best_epoch = 0, 0
    history = []
    save_path = f'best_model_{exp_id}.pt'
    total_start = time.time()
    
    for epoch in range(EPOCHS):
        start = time.time()
        train_loss, train_f1 = train_epoch(model, train_loader, criterion, optimizer, scheduler, device)
        val_loss, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        elapsed = time.time() - start
        history.append({'epoch': epoch+1, 'train_loss': train_loss, 'train_f1': train_f1,
                        'val_loss': val_loss, 'val_f1': val_f1})
        marker = ''
        if val_f1 > best_f1:
            best_f1 = val_f1; best_epoch = epoch + 1
            torch.save(model.state_dict(), save_path)
            marker = ' << BEST'
        print(f'  Epoch {epoch+1} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | {elapsed:.0f}s{marker}')
    
    total_time = (time.time() - total_start) / 60
    
    # Best Model 평가
    model.load_state_dict(torch.load(save_path, weights_only=True))
    _, final_f1, final_preds, final_trues = evaluate(model, val_loader, criterion, device)
    class_names = [id2label[i] for i in range(5)]
    report = classification_report(final_trues, final_preds, target_names=class_names, digits=4, output_dict=True)
    normal_f1 = report['일반 대화']['f1-score']
    
    print(f'\n  >> Best Epoch: {best_epoch} | Val F1: {best_f1:.4f} | 일반대화 F1: {normal_f1:.4f} | {total_time:.1f}분')
    print(classification_report(final_trues, final_preds, target_names=class_names, digits=4))
    
    # Test 추론
    model.eval()
    test_preds = []
    with torch.no_grad():
        for batch in test_loader:
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            test_preds.extend(logits.argmax(-1).cpu().numpy())
    test_classes = [str(id2label[p]) for p in test_preds]
    test_normal = sum(1 for c in test_classes if c == '일반 대화')
    print(f'  >> Test 일반대화: {test_normal}건 / 100건')
    print(f'  >> Test 분포: {pd.Series(test_classes).value_counts().to_dict()}')
    
    all_results.append({
        'exp_id': exp_id, 'desc': desc,
        'backbone_lr': backbone_lr, 'head_lr': head_lr, 'ls': ls, 'pooling': pooling,
        'val_f1': round(best_f1, 4), 'normal_val_f1': round(normal_f1, 4),
        'test_normal': test_normal, 'best_epoch': best_epoch,
        'time_min': round(total_time, 1), 'history': history,
    })
    
    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\n{"="*70}')
print('2단계 Ablation Test 완료!')
print(f'{"="*70}')

In [ ]:
# ============================================================
# Cell 6: 결과 비교 + 분석
# ============================================================

comp = pd.DataFrame([{k:v for k,v in r.items() if k != 'history'} for r in all_results])
comp = comp.sort_values('val_f1', ascending=False)

print('\n' + '=' * 70)
print('2단계 Ablation Test 결과 (Val F1 내림차순)')
print('=' * 70)
print(comp[['exp_id','desc','val_f1','normal_val_f1','test_normal','best_epoch','time_min']].to_string(index=False))

# --- 각 비교 그룹별 분석 ---
print('\n' + '=' * 70)
print('비교 분석')
print('=' * 70)

# LR 비교
lr_group = comp[comp['exp_id'].isin(['M2-A','M2-B','M2-C'])].sort_values('val_f1', ascending=False)
print('\n[LR 전략] 차등 학습률이 필요한가?')
for _, r in lr_group.iterrows():
    print(f'  {r["exp_id"]} {r["desc"]}: Val F1={r["val_f1"]:.4f}, Test 일반={r["test_normal"]}건')
best_lr = lr_group.iloc[0]
print(f'  >> 결론: {best_lr["desc"]} 선택')

# LS 비교 (M2-B vs M2-E)
print('\n[Label Smoothing] 효과가 있는가?')
for eid in ['M2-B', 'M2-E']:
    r = comp[comp['exp_id']==eid]
    if len(r) > 0:
        r = r.iloc[0]
        print(f'  {r["exp_id"]} {r["desc"]}: Val F1={r["val_f1"]:.4f}, Test 일반={r["test_normal"]}건')

# Pooling 비교 (M2-B vs M2-G)
print('\n[Pooling] Mean vs [CLS]?')
for eid in ['M2-B', 'M2-G']:
    r = comp[comp['exp_id']==eid]
    if len(r) > 0:
        r = r.iloc[0]
        print(f'  {r["exp_id"]} {r["desc"]}: Val F1={r["val_f1"]:.4f}, Test 일반={r["test_normal"]}건')

# CSV 저장
comp.to_csv('ablation_strategy_results.csv', index=False)
print('\n결과 저장: ablation_strategy_results.csv')

In [ ]:
# ============================================================
# Cell 7: 시각화 — 그룹별 비교 차트
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# LR 비교
lr_ids = ['M2-A', 'M2-B', 'M2-C']
lr_data = comp[comp['exp_id'].isin(lr_ids)].set_index('exp_id').loc[lr_ids]
axes[0].bar(lr_data['desc'], lr_data['val_f1'], color=['#94a3b8','#3b82f6','#94a3b8'])
axes[0].set_title('LR 전략 비교', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Val F1')
axes[0].set_ylim(lr_data['val_f1'].min()-0.01, lr_data['val_f1'].max()+0.005)
for i, v in enumerate(lr_data['val_f1']): axes[0].text(i, v+0.001, f'{v:.4f}', ha='center', fontsize=10)

# LS 비교
ls_ids = ['M2-B', 'M2-E']
ls_data = comp[comp['exp_id'].isin(ls_ids)].set_index('exp_id').loc[ls_ids]
axes[1].bar(['LS=0.0 (M2-B)', 'LS=0.1 (M2-E)'], ls_data['val_f1'], color=['#94a3b8','#f59e0b'])
axes[1].set_title('Label Smoothing 비교', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Val F1')
axes[1].set_ylim(ls_data['val_f1'].min()-0.01, ls_data['val_f1'].max()+0.005)
for i, v in enumerate(ls_data['val_f1']): axes[1].text(i, v+0.001, f'{v:.4f}', ha='center', fontsize=10)

# Pooling 비교
pool_ids = ['M2-B', 'M2-G']
pool_data = comp[comp['exp_id'].isin(pool_ids)].set_index('exp_id').loc[pool_ids]
axes[2].bar(['Mean (M2-B)', '[CLS] (M2-G)'], pool_data['val_f1'], color=['#10b981','#94a3b8'])
axes[2].set_title('Pooling 비교', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Val F1')
axes[2].set_ylim(pool_data['val_f1'].min()-0.01, pool_data['val_f1'].max()+0.005)
for i, v in enumerate(pool_data['val_f1']): axes[2].text(i, v+0.001, f'{v:.4f}', ha='center', fontsize=10)

plt.suptitle('2단계 Ablation: 학습 전략 비교', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('ablation_strategy_charts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 8: Test 일반대화 예측 수 비교
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))
sorted_comp = comp.sort_values('test_normal', ascending=True)
colors = ['#3b82f6' if eid == 'M2-B' else '#94a3b8' for eid in sorted_comp['exp_id']]
ax.barh(sorted_comp['exp_id'] + ' ' + sorted_comp['desc'], sorted_comp['test_normal'], color=colors)
ax.axvline(x=100, color='red', linestyle='--', alpha=0.5, label='목표 (100건)')
ax.set_xlabel('Test 일반대화 예측 수')
ax.set_title('Test 일반대화 탐지율 비교', fontsize=13, fontweight='bold')
ax.legend()
for i, v in enumerate(sorted_comp['test_normal']):
    ax.text(v+1, i, f'{v}건', va='center', fontsize=11)
plt.tight_layout()
plt.savefig('ablation_test_normal.png', dpi=150, bbox_inches='tight')
plt.show()